# imports

In [1]:
import duckdb
import pandas as pd
import numpy as np

# connection and load

In [2]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

In [3]:
ASSET = 'BTC'
INTERVAL = '1h'

In [4]:
query = f"""
    SELECT * 
    FROM gold_ml_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

# dataframe

In [5]:
df = conn.execute(query).df()

# datetime index 

In [6]:
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

In [7]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,open,high,low,close,volume,daily_volatility,...,returns_5d,returns_10d,returns_20d,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low
date,,,,,,,,,,,,,,,,,,,,,
2020-04-02 17:00:00,BTC,Crypto,Bybit,1h,6908.5,7230.5,6908.5,7081.5,9009.751,322.0,...,0.064647,0.066571,0.116076,0.024733,0.045471,0.537267,6908.5,1552.213,6911.0,6782.5
2020-04-02 18:00:00,BTC,Crypto,Bybit,1h,7081.5,7085.0,6729.5,6787.0,4520.872,355.5,...,0.022062,0.021446,0.042310,-0.042477,0.052380,0.161744,7081.5,9009.751,7230.5,6908.5
2020-04-02 19:00:00,BTC,Crypto,Bybit,1h,6787.0,6863.5,6723.0,6796.5,1274.324,140.5,...,0.003099,0.023723,0.023261,0.001399,0.020672,0.523132,6787.0,4520.872,7085.0,6729.5


# dropping cols and cleaning

In [8]:
cols_to_drop = ['asset_symbol', 'asset_class', 'exchange', 'interval','returns_1d', 'returns_5d', 'returns_10d', 'returns_20d', 'log_returns']

In [9]:
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 39


In [10]:
df.head(5)

,open,high,low,close,volume,daily_volatility,sma_7,sma_30,rsi_14,macd,...,obv,vwap,volume_sma_20,volume_ratio,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low
date,,,,,,,,,,,,,,,,,,,,,
2020-04-02 17:00:00,6908.5,7230.5,6908.5,7081.5,9009.751,322.0,6790.000000,6524.483333,85.656499,130.772825,...,22421.190,6868.681260,1280.33705,7.037015,0.045471,0.537267,6908.5,1552.213,6911.0,6782.5
2020-04-02 18:00:00,7081.5,7085.0,6729.5,6787.0,4520.872,355.5,6805.857143,6543.566667,58.856604,123.436911,...,17900.318,6874.294946,1431.28590,3.158609,0.052380,0.161744,7081.5,9009.751,7230.5,6908.5
2020-04-02 19:00:00,6787.0,6863.5,6723.0,6796.5,1274.324,140.5,6826.571429,6563.550000,59.298990,117.040547,...,19174.642,6877.873019,1426.86415,0.893094,0.020672,0.523132,6787.0,4520.872,7085.0,6729.5
2020-04-02 20:00:00,6796.5,6799.0,6644.0,6738.0,1411.383,155.0,6840.500000,6580.166667,55.352126,106.028688,...,17763.259,6876.222775,1443.52075,0.977737,0.023004,0.606452,6796.5,1274.324,6863.5,6723.0
2020-04-02 21:00:00,6738.0,6843.0,6711.5,6825.5,1056.962,131.5,6847.642857,6600.233333,59.675391,103.172917,...,18820.221,6877.830546,1465.31215,0.721322,0.019266,0.866920,6738.0,1411.383,6799.0,6644.0


# to create target 

In [11]:
df['target_1h'] = df['close'].pct_change().shift(-1)

# dropping the nan from target 

In [13]:
df = df.dropna(subset=['target_1h'])

In [14]:
df[['close', 'target_1h']].tail()

,close,target_1h
date,,
2026-05-06 07:00:00,81332.1,0.007196
2026-05-06 08:00:00,81917.4,0.000601
2026-05-06 09:00:00,81966.6,0.003373
2026-05-06 10:00:00,82243.1,0.002958
2026-05-06 11:00:00,82486.4,-0.003654


In [15]:
df = df.dropna()
print(f"Final data shape after dropping NaNs: {df.shape}")

Final data shape after dropping NaNs: (53395, 40)


In [16]:
df['target_direction'] = (df['target_1h'] > 0).astype(int)
balance = df['target_direction'].value_counts(normalize=True) * 100
print("Class Balance:")
print(balance)

Class Balance:
target_direction
1    50.632082
0    49.367918
Name: proportion, dtype: float64
